In [0]:

# Databricks notebook source
# MAGIC %md
# MAGIC # Bronze: synthetic members and transactions
# MAGIC Generates a reproducible, high-volume core-banking dataset to test the lakehouse at scale.
# MAGIC
# MAGIC - **Members first, then transactions.** Every transaction belongs to a member that exists, so there are no orphan rows downstream.
# MAGIC - **Eligibility-aware dates.** A member's transactions only start once they could legally hold the account: the later of 2009 (TFSA launch), the year they turned 18, and the year they became a resident. FHSA transactions also start no earlier than April 1, 2023.
# MAGIC - **Reproducible.** Every random column uses a fixed seed and the partition count is fixed, so each run produces identical data.
# MAGIC - **Separate from test fixtures.** Synthetic members use IDs from 1000 up. IDs 1 to 6 are the hand-planted test members in Postgres and are never touched here.
# MAGIC - **Raw on purpose.** Bronze keeps the source system's labels (`DEPOSIT` / `WITHDRAWAL`). Silver conforms them to `contribution` / `withdrawal`.
 
# COMMAND ----------
 
from pyspark.sql.functions import (
    broadcast, current_timestamp, expr, lit, rand, round as spark_round,
)
 
# ---- Config ----
NUM_MEMBERS = 500
NUM_TRANSACTIONS = 100_000
NUM_PARTITIONS = 8                 # fixed so seeded rand() returns the same values on every run
MEMBER_ID_START = 1000             # IDs 1-6 are reserved for the planted test members in Postgres
SOURCE_SYSTEM = "synthetic_core_banking_v1"
 
SCHEMA = "personal_finance_bronze"
MEMBERS_TABLE = f"{SCHEMA}.members_raw"
TXNS_TABLE = f"{SCHEMA}.member_transactions_raw"
 
MEMBER_ID_END = MEMBER_ID_START + NUM_MEMBERS - 1
 
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## 1. Members
# MAGIC Same shape as the Postgres `members` table: birth year, province and the year they became a resident.
 
# COMMAND ----------
 
df_members = (
    spark.range(MEMBER_ID_START, MEMBER_ID_END + 1, 1, NUM_PARTITIONS)
    .withColumnRenamed("id", "member_id")
    .withColumn("first_name", lit("Synthetic"))
    .withColumn("last_name", expr("concat('Member', member_id)"))
 
    # Born 1950-2006, so everyone is 18 or older by 2024
    .withColumn("birth_year", (rand(11) * 57).cast("int") + 1950)
 
    .withColumn(
        "province",
        expr("element_at(array('BC','AB','SK','MB','ON','QC','NB','NS','PE','NL'), "
             "cast(rand(12) * 10 as int) + 1)"),
    )
 
    # 80% resident since birth; 20% newcomers who arrived between 2000 and 2025
    .withColumn(
        "resident_since",
        expr("CASE WHEN rand(13) < 0.80 THEN birth_year "
             "ELSE greatest(birth_year, 2000 + cast(rand(14) * 26 as int)) END"),
    )
    .withColumn("source_system", lit(SOURCE_SYSTEM))
    .withColumn("ingest_timestamp", current_timestamp())
)
 
(
    df_members.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(MEMBERS_TABLE)
)
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## 2. Transactions
# MAGIC Random values are drawn first, then each transaction joins to its member to place its date inside the member's eligible window.
 
# COMMAND ----------
 
members = spark.table(MEMBERS_TABLE).select(
    "member_id",
    # First day the member could hold a TFSA or RRSP in this dataset
    expr("make_date(greatest(2009, birth_year + 18, resident_since), 1, 1)").alias("eligible_from"),
)
 
df_random = (
    spark.range(0, NUM_TRANSACTIONS, 1, NUM_PARTITIONS)
    .withColumnRenamed("id", "txn_id")
    .withColumn("member_id", (rand(1) * NUM_MEMBERS).cast("int") + MEMBER_ID_START)
 
    # Account mix: 50% TFSA, 35% RRSP, 15% FHSA
    .withColumn("_r_type", rand(2))
    .withColumn(
        "account_type",
        expr("CASE WHEN _r_type < 0.50 THEN 'TFSA' "
             "WHEN _r_type < 0.85 THEN 'RRSP' ELSE 'FHSA' END"),
    )
    .withColumn("_r_date", rand(3))
 
    # 60% deposits, 40% withdrawals, in the source system's raw labels
    .withColumn("transaction_type", expr("CASE WHEN rand(4) < 0.60 THEN 'DEPOSIT' ELSE 'WITHDRAWAL' END"))
 
    # $10 to $2,510, stored as money (decimal), not float
    .withColumn("transaction_amount", spark_round(rand(5) * 2500 + 10, 2).cast("decimal(12,2)"))
)
 
# Broadcast join: the small members table is copied to every worker, so the
# transactions keep their partitioning and the seeded values stay stable
df_txns = (
    df_random.join(broadcast(members), "member_id")
    .withColumn(
        "_start",
        expr("CASE WHEN account_type = 'FHSA' "
             "THEN greatest(eligible_from, DATE'2023-04-01') ELSE eligible_from END"),
    )
    .withColumn(
        "txn_date",
        expr("date_add(_start, cast(_r_date * datediff(DATE'2025-12-31', _start) as int))"),
    )
    .withColumn("source_system", lit(SOURCE_SYSTEM))
    .withColumn("ingest_timestamp", current_timestamp())
    .select(
        "txn_id", "member_id", "account_type", "txn_date", "transaction_type",
        "transaction_amount", "source_system", "ingest_timestamp",
    )
)
 
(
    df_txns.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TXNS_TABLE)
)
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## 3. Validation
# MAGIC Fails the notebook if the data breaks any rule the downstream layers depend on.
 
# COMMAND ----------
 
m = spark.table(MEMBERS_TABLE)
t = spark.table(TXNS_TABLE)
 
member_count = m.count()
assert member_count == NUM_MEMBERS, f"Expected {NUM_MEMBERS} members, got {member_count}"
 
txn_count = t.count()
assert txn_count == NUM_TRANSACTIONS, f"Expected {NUM_TRANSACTIONS} transactions, got {txn_count}"
 
orphans = t.join(m, "member_id", "left_anti").count()
assert orphans == 0, f"{orphans} transactions have no matching member"
 
bad_types = t.filter("account_type NOT IN ('TFSA', 'RRSP', 'FHSA')").count()
assert bad_types == 0, f"{bad_types} rows have an invalid account_type"
 
early_fhsa = t.filter("account_type = 'FHSA' AND txn_date < DATE'2023-04-01'").count()
assert early_fhsa == 0, f"{early_fhsa} FHSA transactions are dated before the account existed"
 
ineligible = (
    t.join(m, "member_id")
    .filter("year(txn_date) < greatest(2009, birth_year + 18, resident_since)")
    .count()
)
assert ineligible == 0, f"{ineligible} transactions are dated before the member was eligible"
 
reserved = m.filter("member_id < 1000").count()
assert reserved == 0, "Synthetic members must not use the reserved test-member IDs"

 
print(f"Ingested {member_count:,} members and {txn_count:,} transactions. All checks passed.")
 